Jupyter notebook for data analysis

# 1. Prepare behavior data for later process

## 1.1 Preprocessing of behavior files in Matlab

run Analysis/behavior_process_matlab/behavior_pipeline_full.m (see [README](./README.md) for details)


## 1.2 Go over every behavior session

In [ ]:
%matplotlib inline
import pickle

import numpy as np
import pandas as pd

root_dir = {}

# 4 groups in total: 
# 1. animals learnt the task at late learning stage (experts)
# 2. animals learnt the task at early learning stage (to-be experts)
# 3. animals that failed in the end at early learning stage
# 4. animals that failed in the end at late learning stage
root_dir['LateLearnt'] = r'GithubData\Late_Learner'
root_dir['EarlyLearnt'] = r'Github\Early_Learner'
root_dir['LateNonlearner'] = r'Github\Late_nonlearner'
root_dir['EarlyNonlearner'] = r'Github\Early_nonlearner'

keyList = root_dir.keys()

In [ ]:
# process the behavioral data
from behavioral_pipeline import *
for key in root_dir.keys():
    beh_sum = GoNogoBehaviorSum(root_dir[key])
    #matplotlib.use('Agg')
    beh_sum.process_singleSession(ifrun=True)

# 2. Preprocessing of fluorescent data

## 2.1 calculate dF/F, read sessions into dataframe for later analysis

### Experts in late learning stage

In [ ]:
# initialize session list
from fluorescent_pipeline import *

In [ ]:
fluo_summary = {}
ADT = {}
JUV = {}
group_label = {}

In [ ]:
key = "LateLearnt"
fluo_summary[key] = fluoSum(root_dir[key])
fluo_summary[key].process_single_session()

# optional: plot PSTH for every neuron
#fluo_summary.cell_plots()

# set the adolescents and adults groups        
for ii in range(len(fluo_summary[key].data_df['age'])):
    if fluo_summary[key].data_df['age'][ii] == 'TRA':
        fluo_summary[key].data_df['age'][ii] = 'ADT'

c2 = np.logical_and(fluo_summary[key].data_df['age'] == 'ADT', fluo_summary[key].data_df['with_imaging'])
c3 = np.logical_and(fluo_summary[key].data_df['age'] == 'JUV',fluo_summary[key].data_df['with_imaging'])

ADT[key] = fluo_summary[key].data_df[c2]
JUV[key] = fluo_summary[key].data_df[c3]
group_label[key] = key

### Experts in early learning stage

In [ ]:
key = "EarlyLearnt"
fluo_summary[key] = fluoSum(root_dir[key])
fluo_summary[key].process_single_session()

# optional: plot PSTH for every neuron
#fluo_summary.cell_plots()

# set the adolescents and adults groups        
for ii in range(len(fluo_summary[key].data_df['age'])):
    if fluo_summary[key].data_df['age'][ii] == 'TRA':
        fluo_summary[key].data_df['age'][ii] = 'ADT'

c2 = np.logical_and(fluo_summary[key].data_df['age'] == 'ADT', fluo_summary.data_df['with_imaging'])
c3 = np.logical_and(fluo_summary[key].data_df['age'] == 'JUV',fluo_summary.data_df['with_imaging'])

ADT[key] = fluo_summary[key].data_df[c2]
JUV[key] = fluo_summary[key].data_df[c3]
group_label[key] = key

### non-learners in late stage

In [ ]:
key = "LateNonlearner"
fluo_summary[key] = fluoSum(root_dir[key])
fluo_summary[key].process_single_session()

# optional: plot PSTH for every neuron
#fluo_summary.cell_plots()

# set the adolescents and adults groups        
for ii in range(len(fluo_summary[key].data_df['age'])):
    if fluo_summary[key].data_df['age'][ii] == 'TRA':
        fluo_summary[key].data_df['age'][ii] = 'ADT'

c2 = np.logical_and(fluo_summary[key].data_df['age'] == 'ADT', fluo_summary.data_df['with_imaging'])
c3 = np.logical_and(fluo_summary[key].data_df['age'] == 'JUV',fluo_summary.data_df['with_imaging'])

ADT[key] = fluo_summary[key].data_df[c2]
JUV[key] = fluo_summary[key].data_df[c3]
group_label[key] = key

### non-learners in early stage

In [ ]:
key = "EarlyNonlearner"
fluo_summary[key] = fluoSum(root_dir[key])
fluo_summary[key].process_single_session()

# optional: plot PSTH for every neuron
#fluo_summary.cell_plots()

# set the adolescents and adults groups        
for ii in range(len(fluo_summary[key].data_df['age'])):
    if fluo_summary[key].data_df['age'][ii] == 'TRA':
        fluo_summary[key].data_df['age'][ii] = 'ADT'

c2 = np.logical_and(fluo_summary[key].data_df['age'] == 'ADT', fluo_summary.data_df['with_imaging'])
c3 = np.logical_and(fluo_summary[key].data_df['age'] == 'JUV',fluo_summary.data_df['with_imaging'])

ADT[key] = fluo_summary[key].data_df[c2]
JUV[key] = fluo_summary[key].data_df[c3]
group_label[key] = key

## 2.2 Multiple linear regression - determine the effect of running on dF/F

In [0]:
# multiple linear regression for each session
key='LateLearnt'
fluo_summary[key].MLR_session()

In [ ]:
%matplotlib inline
#fluo_summary[key].MLR_session()

key = "LateLearnt"
fluo_summary[key].MLR_summary(ADT[key], JUV[key], group_label[key])

In [ ]:
# comparison of latelearnt and early learnt group
#stats across early and late learning stage
import os
from scipy.stats import wilcoxon, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from scipy.stats import median_abs_deviation

LateRunningPath = os.path.join(root_dir['LateLearnt'], r'Summary\fluo\RunningVariance_session.pickle')
EarlyRunningPath = os.path.join(root_dir['EarlyLearnt'], r'Summary\fluo\RunningVariance_session.pickle')

with open(LateRunningPath,"rb") as pickle_file:
    # Load the data from the pickle file
    latevar = pickle.load(pickle_file)
    
with open(EarlyRunningPath,"rb") as pickle_file:
    # Load the data from the pickle file
    earlyvar = pickle.load(pickle_file)

stats, p_learningJUV1 = mannwhitneyu(earlyvar['aveRunningSesJUV1'], latevar['aveRunningSesJUV1'])
stats, p_learningJUV2 = mannwhitneyu(earlyvar['aveRunningSesJUV2'], latevar['aveRunningSesJUV2'])
stats, p_learningADT1 = mannwhitneyu(earlyvar['aveRunningSesADT1'], latevar['aveRunningSesADT1'])
stats, p_learningADT2 = mannwhitneyu(earlyvar['aveRunningSesADT2'], latevar['aveRunningSesADT2'])
p_list = np.array([p_learningJUV1, p_learningJUV2, p_learningADT1, p_learningADT2])
q_list = multipletests(p_list, method='fdr_bh')[1]

print("Median of ADT 1-2 s, early:", np.median(earlyvar['aveRunningSesADT2']))
print("MAD of ADT 1-2 s, early:", median_abs_deviation(earlyvar['aveRunningSesADT2']))
print ("Median of ADT 1-2 s, late:", np.median(latevar['aveRunningSesADT2']))
print("MAD of ADT 1-2 s, late:", median_abs_deviation(latevar['aveRunningSesADT2']))
print("Median of JUV 1-2 s, early:", np.median(earlyvar['aveRunningSesJUV2']))
print("MAD of JUV 1-2 s, early:", median_abs_deviation(earlyvar['aveRunningSesJUV2']))
print("Median of JUV 1-2 s, late:", np.median(latevar['aveRunningSesJUV2']))
print("MAD of JUV 1-2 s, late:", median_abs_deviation(latevar['aveRunningSesJUV2']))

stages = ['JUV 0-1s', 'JUV 1-2s', 'ADT 0-1s', 'ADT 1-2s']
for idx,stage in enumerate(stages):
    print("Learning stage" + stage + f" p-value {p_list[idx]:.8f}, adjusted p-value {p_list[idx]:.8f} ")
data = {'stats': ['JUV0-1', 'JUV1-2', 'ADT0-1', 'ADT1-2'],
        'p_values': p_list,
        'adj_p_values': q_list}
dataDF = pd.DataFrame(data)
SummaryFolder = os.path.join(root_dir['LateLearnt'], r'Summary/ComparisonXLearning')
if not os.path.exists(SummaryFolder):
    os.makedirs(SummaryFolder)
dataDF.to_csv(os.path.join(SummaryFolder, 'runningVariance_xlearning_session.csv'))


## Decoding analysis

In [ ]:
# decoding analysis
classifier = 'SVC'
spec = ''  # vanilla decoding
n_predictors = 14
for key in keyList:
    fluo_summary[key]. decoding_session(n_predictors, classifier, spec)
 
spec = 'noRun'
fluo_summary['LateLearnt']. decoding_session(n_predictors, classifier, spec)

In [ ]:
# compare decoding accuracy across learning for experts animals
# load the data
%matplotlib inline
lateLearntFile = os.path.join(root_dir['LateLearnt'],'Summary','fluo','decoding_accuracy_1sWindow.pickle')
earlyLearntFile = os.path.join(root_dir['EarlyLearnt'],'Summary','fluo','decoding_accuracy_1sWindow.pickle')

lateNonlearnerFile = os.path.join(root_dir['LateNonlearner'],'Summary','fluo','decoding_accuracy_1sWindow_allSession.pickle')
earlyNonlearnerFile = os.path.join(root_dir['EarlyNonlearner'],'Summary','fluo','decoding_accuracy_1sWindow_allSession.pickle')

with open(lateNonlearnerFile, 'rb') as f:
    lateNonlearnerResult = pickle.load(f)
    f.close()
with open(earlyNonlearnerFile, 'rb') as f:
    earlyNonlearnerResult = pickle.load(f)
    f.close()
with open(lateLearntFile, 'rb') as f:
    lateLearntResult = pickle.load(f)
    f.close()
with open(earlyLearntFile, 'rb') as f:
    earlyLearntResult = pickle.load(f)
    f.close()
    
# compare early and late learnt for decoding accuracy
p_MWtest1 = mannwhitneyu(earlyLearntResult['ADT1'], lateLearntResult['ADT1'])[1]
p_MWtest2 = mannwhitneyu(earlyLearntResult['ADT2'], lateLearntResult['ADT2'])[1]
p_MWtest3 = mannwhitneyu(earlyLearntResult['JUV1'], lateLearntResult['JUV1'])[1]
p_MWtest4 = mannwhitneyu(earlyLearntResult['JUV2'], lateLearntResult['JUV2'])[1]
p_list = np.array([p_MWtest1, p_MWtest2, p_MWtest3, p_MWtest4])
q_list = multipletests(p_list, method = 'fdr_bh')[1]
groups = ['ADT 0-1s', 'ADT 1-2s', 'JUV 0-1s', 'JUV 1-2s']
print("Comparison between early and late learnt animals")
for idx,group in enumerate(groups):
    print("Decoding accuracy for "+group+ f" p-value: {p_list[idx]:.4e}, adjusted p-value: {q_list[idx]:.4e}")
print("=======================================================================\n") 
# compare non-learn with early learnt
combinedResult = {}
for key in lateLearntResult.keys():
    combinedResult[key] = np.concatenate((earlyNonlearnerResult[key],lateNonlearnerResult[key]))
p_MWtest1 = mannwhitneyu(earlyLearntResult['ADT1'], combinedResult['ADT1'])[1]
p_MWtest2 = mannwhitneyu(earlyLearntResult['ADT2'], combinedResult['ADT2'])[1]
p_MWtest3 = mannwhitneyu(earlyLearntResult['JUV1'], combinedResult['JUV1'])[1]
p_MWtest4 = mannwhitneyu(earlyLearntResult['JUV2'], combinedResult['JUV2'])[1]
p_list = np.array([p_MWtest1, p_MWtest2, p_MWtest3, p_MWtest4])
q_list = multipletests(p_list, method = 'fdr_bh')[1]
groups = ['ADT 0-1s', 'ADT 1-2s', 'JUV 0-1s', 'JUV 1-2s']
print("Comparison between non learns and early learnt animals")
for idx,group in enumerate(groups):
    print("Decoding accuracy for "+group+ f" p-value: {p_list[idx]:.4e}, adjusted p-value: {q_list[idx]:.4e}")
    
print("=======================================================================\n") 

p_MWtest1 = mannwhitneyu(lateLearntResult['ADT1'], combinedResult['ADT1'])[1]
p_MWtest2 = mannwhitneyu(lateLearntResult['ADT2'], combinedResult['ADT2'])[1]
p_MWtest3 = mannwhitneyu(lateLearntResult['JUV1'], combinedResult['JUV1'])[1]
p_MWtest4 = mannwhitneyu(lateLearntResult['JUV2'], combinedResult['JUV2'])[1]
p_list = np.array([p_MWtest1, p_MWtest2, p_MWtest3, p_MWtest4])
q_list = multipletests(p_list, method = 'fdr_bh')[1]
groups = ['ADT 0-1s', 'ADT 1-2s', 'JUV 0-1s', 'JUV 1-2s']
print("Comparison between non learns and late learnt animals")
for idx,group in enumerate(groups):
    print("Decoding accuracy for "+group+ f" p-value: {p_list[idx]:.4e}, adjusted p-value: {q_list[idx]:.4e}")
    
    

In [ ]:
# now try no cap, all neurons included for every session
%matplotlib inline
import os
from scipy.stats import wilcoxon, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import pandas as pd
from scipy.stats import median_abs_deviation
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multicomp import MultiComparison
from scipy import stats

lateNonlearnerFile = os.path.join(root_dir['LateNonlearner'],'Summary','fluo','decoding_accuracy_1sWindow_allSession.pickle')
earlyNonlearnerFile = os.path.join(root_dir['EarlyNonlearner'],'Summary','fluo','decoding_accuracy_1sWindow_allSession.pickle')
nIdx = 100

with open(lateNonlearnerFile, 'rb') as f:
    lateResult = pickle.load(f)
    f.close()
with open(earlyNonlearnerFile, 'rb') as f:
    earlyResult = pickle.load(f)
    f.close()
    
# combine the result
combinedResult = {}
for key in lateResult.keys():
    combinedResult[key] = np.concatenate((earlyResult[key],lateResult[key]))
    globals()[key] = combinedResult[key]
    
p_MWtest1 = mannwhitneyu(ADT1, JUV1)[1]
p_MWtest2 = mannwhitneyu(ADT2, JUV2)[1]
p_list = np.array([p_MWtest1, p_MWtest2])
q_list = multipletests(p_list, method = 'fdr_bh')[1]
print("Mannwhitney results between adults and adolescents in 0-1 and 1-2s time window:")
print(p_list)

# plot the combined result
stat,p_ctrlADT1 = wilcoxon(ADT1, ADT1_Ctrl)
stat,p_ctrlADT2 = wilcoxon(ADT2, ADT2_Ctrl)
stat,p_ctrlJUV1 = wilcoxon(JUV1, JUV1_Ctrl)
stat,p_ctrlJUV2 = wilcoxon(JUV2, JUV2_Ctrl)
p_list_ctrl = np.array([p_ctrlADT1, p_ctrlADT2, p_ctrlJUV1, p_ctrlJUV2])
q_list_ctrl = multipletests(p_list_ctrl, method = 'fdr_bh')[1]

# print the p values
groups = ['ADT 0-1s', 'ADT 1-2s', 'JUV 0-1s', 'JUV 1-2s']
for idx,group in enumerate(groups):
    print("Decoding accuracy for "+group+ f" p-value: {p_list_ctrl[idx]:.4e}, adjusted p-value: {q_list_ctrl[idx]:.4e}")
# save the pvalues
data = {'groups':['Age0-1', 'Age1-2', 'ADT0-1', 'ADT1-2','JUV0-1', 'JUV1-2'],
        'p_values': np.concatenate((p_list, p_list_ctrl)),
        'adj_p_values': np.concatenate((q_list, q_list_ctrl))}
dataDF = pd.DataFrame(data)
#dataDF.to_csv(os.path.join(saveFigPath, 'stats_for_aveDecoding_n='+str((nIdx+1)*10)+'.csv'))
print("Median of ADT 0-1s:", np.median(ADT1))
print("MAD of ADT 0-1s:", median_abs_deviation(ADT1))
print ("Median of ADT 1-2s:", np.median(ADT2))
print("MAD of ADT 1-2s:", median_abs_deviation(ADT2))
print("Median of JUV 0-1s:", np.median(JUV1))
print("MAD of JUV 0-1s:", median_abs_deviation(JUV1))
print("Median of JUV 1-2s:", np.median(JUV2))
print("MAD of JUV 1-2s:", median_abs_deviation(JUV2))

print("Median of ADT ctrl 0-1s:", np.median(ADT1_Ctrl))
print("MAD of ADT ctrl 0-1s:", median_abs_deviation(ADT1_Ctrl))
print ("Median of ADT ctrl 1-2s:", np.median(ADT2_Ctrl))
print("MAD of ADT ctrl 1-2s:", median_abs_deviation(ADT2_Ctrl))
print("Median of JUV ctrl 0-1s:", np.median(JUV1_Ctrl))
print("MAD of JUV ctrl 0-1s:", median_abs_deviation(JUV1_Ctrl))
print("Median of JUV ctrl 1-2s:", np.median(JUV2_Ctrl))
print("MAD of JUV ctrl 1-2s:", median_abs_deviation(JUV2_Ctrl))
        
x=np.array([1,2])
adtColor = (83/255,187/255,244/255)
juvColor = (255/255,67/255,46/255)
averageBoxPlot = StartPlots()
averageBoxPlot.ax.set_title('Decoding Accuracy and Control by Age Group, '+str((nIdx+1)*10)+' cells')
averageBoxPlot.ax.set_ylim([0.45,1])
boxplot = averageBoxPlot.ax.boxplot([ADT1, ADT2],
                                    positions= x - 0.3,widths=0.1,patch_artist=True, showfliers=False,
                                    boxprops=dict(facecolor=adtColor, edgecolor=adtColor),
                                    whiskerprops=dict(color=adtColor),
                                    capprops=dict(color=adtColor),
                                    medianprops=dict(marker='o', color='white', markersize=5))

boxplot =averageBoxPlot.ax.boxplot([ADT1_Ctrl,ADT2_Ctrl],
                                   positions=x - 0.1, widths=0.1,patch_artist=True, showfliers=False,
                                   boxprops=dict(facecolor=(0.7, 0.7, 0.7), edgecolor=(0.7, 0.7, 0.7)),
                                    whiskerprops=dict(color=(0.7, 0.7, 0.7)),
                                    capprops=dict(color=(0.7, 0.7, 0.7)),
                                    medianprops=dict(marker='o', color='white', markersize=5))

averageBoxPlot.ax.set_xticks(x, ['0-1s', '1-2s'])

# plot JUV
# plot data and ctrl
boxplot = averageBoxPlot.ax.boxplot([JUV1, JUV2],
                                    positions=x + 0.1,widths=0.1,patch_artist=True, showfliers=False,
                                    boxprops=dict(facecolor=juvColor, edgecolor=juvColor),
                                    whiskerprops=dict(color=juvColor),
                                    capprops=dict(color=juvColor),
                                    medianprops=dict(marker='o', color='white', markersize=5))

boxplot = averageBoxPlot.ax.boxplot([JUV1_Ctrl, JUV2_Ctrl],
                                    positions=x + 0.3,widths=0.1, patch_artist=True, showfliers=False,
                                    boxprops=dict(facecolor=(0.7, 0.7, 0.7), edgecolor=(0.7, 0.7, 0.7)),
                                    whiskerprops=dict(color=(0.7, 0.7, 0.7)),
                                    capprops=dict(color=(0.7, 0.7, 0.7)),
                                    medianprops=dict(marker='o', color='white', markersize=5)
                                    )

averageBoxPlot.ax.set_xticks([1, 2], ['0-1s', '1-2s'])
averageBoxPlot.ax.set_ylim([0.3, 1])
plt.show()
SummaryFolder = os.path.join(root_dir['LateLearnt'], r'Summary/ComparisonXLearning')
if not os.path.exists(SummaryFolder):
    os.makedirs(SummaryFolder)
averageBoxPlot.save_plot('Average decoding accuracy boxplot for nonlearners combined.png', 'png', SummaryFolder)
averageBoxPlot.save_plot('Average decoding accuracy boxplot for nonlearners combined.svg', 'svg', SummaryFolder)


In [ ]:
# compare non-learners and learners


## 2.3 Decoding of stimulus information
- decode go or nogo stimulus from dF/F of all neurons 
- decode go or nogo stimulus from dF/F of 10:10:100 neurons (randomly sampled for 20 times and averaged across random samples)

In [ ]:
n_predictors = 14
fluo_summary[key].decoding_session(n_predictors)

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2
from fluorescent_pipeline import *
fluo_summary[key].decoding_summary(ADT[key], JUV[key], group_label[key])

## 2.3 decode individual cue pairs (e.g. cue 5 against 6, and cue 3 against cue 4) 

In [ ]:
cue_list = [[3,4,5,6], [1,2,7,8],[1,3,6,8],[2,3,6,7],[1,4,5,8],[2,4,5,7]]
for cue_pairs in cue_list:
    fluo_summary.decoding_hardeasy_session(n_predictors, cue_pairs)
    fluo_summary.decoding_hardeasy_summary(ADT_late, JUV_late, cue_pairs, group_label)

## 2.4 decoding accuracy of pseudo ensembles (shuffle the data to destroy the noise correlation)

In [ ]:
fluo_summary.pseudo_session()
fluo_summary.pseudo_summray(ADT_late, JUV_late, group_label)

## 2.5 noise correlation analysis (reference to be added)

In [ ]:
fluo_summary.noise_session()
fluo_summary.noise_summary_Valente_2021(ADT_late, JUV_late, group_label)